# 02 — Baseline: full fine-tuning

**Goal:** train DistilBERT on SST-2 updating *all* weights. This is the reference point: every LoRA run is judged against this number.

**Fixed settings for the whole study** (same in every notebook and in `src/train.py`): seed 42, 3 epochs, max length 128, cosine schedule, warmup 0.1, weight decay 0.01, lr 2e-5 for full fine-tuning. Only `mode` and `r` change between runs — otherwise the comparison is meaningless.

In [ ]:
# Setup: run this first in Colab (or skip locally if requirements are installed)
# In Colab, clone the repo so src/ and experiments/ are available:
#   !git clone https://github.com/sujalgiriiitp-source/Lora---reproduction-.git
#   %cd Lora---reproduction-
# Then install dependencies (takes a few minutes the first time):
#   !pip install -q -r requirements.txt
import sys
print(sys.version)
try:
    import torch, transformers, peft, datasets
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
    print("transformers", transformers.__version__, "| peft", peft.__version__, "| datasets", datasets.__version__)
except ImportError as e:
    print("Missing package:", e)
    print("Run: pip install -r requirements.txt")


## 1. Load SST-2

SST-2 is binary sentiment (positive / negative movie reviews). Train: 67,349 examples. Validation: 872. The small validation set matters later — differences under ~1–2 points are noise, not wins.

In [ ]:
from datasets import load_dataset
raw = load_dataset("nyu-mll/glue", "sst2")
print(raw)
print(raw["train"][0])

## 2. Tokenize

DistilBERT tokenizer, truncated to 128 tokens. SST-2 sentences are short, so almost nothing gets truncated — check the length distribution if you doubt it.

In [ ]:
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
def tokenize(batch):
    return tok(batch["sentence"], truncation=True, max_length=128)
train_tok = raw["train"].map(tokenize, batched=True, remove_columns=raw["train"].column_names)
eval_tok = raw["validation"].map(tokenize, batched=True, remove_columns=raw["validation"].column_names)
print(train_tok[0])

## 3. Train — all weights (via the shared script)

The training itself lives in `src/train.py` so the notebook, the script, and the Colab runs all use identical code. This cell runs it. On a Colab T4 this takes ~10–15 minutes for 3 epochs; on CPU, do **not** run the full set — use the smoke-test flags instead (see `SMOKE_TEST_LOG.md`).

In [ ]:
# Full run (GPU). For a quick CPU check, add: --epochs 1 --max-train 200 --max-eval 100 --batch-size 8
!python src/train.py --mode full --epochs 3 --batch-size 32 --output runs/full --run-id full

## 4. Read the result back

`train.py` appends one row to `experiments/runs.csv`. Read it here — the number in the README and the report must come from this file, not from memory.

In [ ]:
import pandas as pd
runs = pd.read_csv("experiments/runs.csv")
print(runs[runs.run_id == "full"][["run_id", "mode", "trainable_params", "accuracy", "f1_macro", "train_time_sec"]].to_string(index=False))

**What to note before moving on:** the full fine-tuning accuracy is the target LoRA has to approach, and its training time is the cost LoRA has to beat. If this baseline looks wrong (e.g. accuracy near 50%, which is chance for binary classification), fix it before running LoRA — a broken baseline makes every comparison meaningless.

**Next:** notebook 03 runs LoRA at r=8 with the same data, seed, and epochs.